In [ ]:
import torchaudio
import torchaudio.transforms as T

def resamplear(ruta_wav, sr_objetivo=22050):
    """
    Carga un .wav, resamplea si es necesario y devuelve numpy array (N,) mono.
    
    Entrada:  ruta al archivo .wav
    Salida:   numpy array (N,)
    """
    waveform, sr = torchaudio.load(ruta_wav)
    
    if sr != sr_objetivo:
        waveform = T.Resample(orig_freq=sr, new_freq=sr_objetivo)(waveform)
    
    # Stereo → mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    return waveform.squeeze(0).numpy()

import os
import numpy as np
import librosa
import soundfile as sf

  MAJO 
    |
    V

In [ ]:
DATASET_PATH  = r"C:\Users\USUARIO\Downloads\UrbanSound8K\UrbanSound8K"
OUTPUT_PATH   = r"C:\Users\USUARIO\Downloads\UrbanSound8K\UrbanSound8K_4sec"
DURACION_SEG  = 4
SAMPLE_RATE   = 22050

AUDIO_ROOT   = os.path.join(DATASET_PATH, "audio")
MUESTRAS_OBJ = DURACION_SEG * SAMPLE_RATE

def ajustar_duracion(audio):
    if len(audio) == MUESTRAS_OBJ:
        return audio
    if len(audio) < MUESTRAS_OBJ:
        repeticiones = int(np.ceil(MUESTRAS_OBJ / len(audio)))
        audio_rep = np.tile(audio, repeticiones)
        return audio_rep[:MUESTRAS_OBJ]
    return audio[:MUESTRAS_OBJ]

def procesar_dataset():
    procesados = 0
    errores = 0
    for fold_num in range(1, 11):
        fold_nombre  = f"fold{fold_num}"
        fold_entrada = os.path.join(AUDIO_ROOT, fold_nombre)
        fold_salida  = os.path.join(OUTPUT_PATH, fold_nombre)
        if not os.path.isdir(fold_entrada):
            print(f"[WARN] No encontrada: {fold_entrada}")
            continue
        os.makedirs(fold_salida, exist_ok=True)
        archivos = [f for f in os.listdir(fold_entrada) if f.lower().endswith(".wav")]
        print(f"\n📁 {fold_nombre} — {len(archivos)} archivos")
        for fname in archivos:
            ruta_entrada = os.path.join(fold_entrada, fname)
            ruta_salida  = os.path.join(fold_salida, fname)
            try:
                audio, sr = librosa.load(ruta_entrada, sr=SAMPLE_RATE, mono=True)
                audio_ajustado = ajustar_duracion(audio)
                sf.write(ruta_salida, audio_ajustado, SAMPLE_RATE)
                procesados += 1
            except Exception as e:
                print(f"  [ERROR] {fname}: {e}")
                errores += 1
        print(f"  ✅ {fold_nombre} completado")
    print(f"\n✅ Procesados: {procesados} | ❌ Errores: {errores}")
    print(f"📂 Guardados en: {OUTPUT_PATH}")

procesar_dataset()